In [1]:
# MEAWAN - Libraries Setup

!pip -q install --upgrade gradio gTTS

import gradio as gr
import hashlib
import json
import html
import tempfile

from datetime import datetime, timezone
from gtts import gTTS

print("MEAWAN is ready!")
print("Gradio version:", gr.__version__)
print("Text-to-Speech is ready!")


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


/usr/local/python/3.14.2/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MEAWAN is ready!
Gradio version: 6.31.0
Text-to-Speech is ready!


In [2]:
import gradio as gr
import hashlib, json, html, tempfile, re
from datetime import datetime, timezone
from gtts import gTTS

AR_URL = 'https://vocal-cocada-7ad1b3.netlify.app'

TASKS = {
 'bag': {'ar': ['افتح الحقيبة','ضع الكتاب داخل الحقيبة','أغلق الحقيبة'], 'en':['Open your bag','Put the book inside your bag','Close your bag']},
 'room': {'ar':['رتب سريرك','ضع الملابس في مكانها','رتب أغراضك','تأكد من نظافة الغرفة'], 'en':['Make your bed','Put your clothes away','Organize your belongings','Check your room']},
 'teeth': {'ar':['أحضر فرشاة الأسنان','ضع المعجون على الفرشاة','نظف أسنانك','اغسل الفرشاة'], 'en':['Get your toothbrush','Apply toothpaste','Brush your teeth','Rinse your toothbrush']}
}
TEXT = {
 'ar': {
  'title':'مِعوان', 'tagline':'مساعدك في كل خطوة نحو الاستقلالية', 'welcome':'مرحبًا بك في مِعوان', 'intro':'اختر المهمة وطريقة المساعدة المناسبة لك.',
  'task':'المهمة اليومية','profile':'طريقة المساعدة','level':'مستوى المساعدة','start':'ابدأ المهمة','next':'أنجزت الخطوة','voice':'استمع للتعليمات','audio':'التعليمات الصوتية',
  'ready':'اختر مهمة واضغط ابدأ المهمة.','done':'أحسنت! أكملت المهمة بنجاح.','step':'الخطوة','of':'من','progress':'التقدم',
  'ar_heading':'مساعد الواقع المعزز','ar_open':'فتح تجربة AR بالكاميرا','history':'لوحة متابعة الأسرة','show':'عرض الإنجازات','verify':'التحقق من سلامة السجلات',
  'records':'سجلات الإنجاز','result':'نتيجة التحقق','empty':'لا توجد إنجازات مسجلة بعد.','valid':'نجح فحص سلامة بصمات السجلات.','invalid':'تم اكتشاف خلل في سلامة السجلات.',
  'tasks':[('تجهيز الحقيبة','bag'),('ترتيب الغرفة','room'),('تنظيف الأسنان','teeth')],
  'profiles':[('إرشادات مبسطة','simple'),('إرشادات مرئية','visual'),('إرشادات نصية','text'),('أزرار كبيرة','motor')],
  'levels':[('مساعدة كاملة','high'),('مساعدة متوسطة','medium'),('مساعدة بسيطة','low')]},
 'en': {
  'title':'MEAWAN','tagline':'Your companion toward independence','welcome':'Welcome to MEAWAN','intro':'Choose your task and preferred guidance.',
  'task':'Daily Task','profile':'Guidance Preference','level':'Assistance Level','start':'Start Task','next':'Step Completed','voice':'Listen to Instructions','audio':'Voice Instructions',
  'ready':'Choose a task and press Start Task.','done':'Well done! You completed the task.','step':'Step','of':'of','progress':'Progress',
  'ar_heading':'Augmented Reality Assistant','ar_open':'Open AR Camera Experience','history':'Family Dashboard','show':'View Achievements','verify':'Verify Records',
  'records':'Achievement Records','result':'Verification Result','empty':'No achievements recorded yet.','valid':'Record hashes passed integrity verification.','invalid':'Record integrity check failed.',
  'tasks':[('Pack My Bag','bag'),('Tidy My Room','room'),('Brush My Teeth','teeth')],
  'profiles':[('Simplified Guidance','simple'),('Visual Guidance','visual'),('Text Guidance','text'),('Large Buttons','motor')],
  'levels':[('Full Assistance','high'),('Moderate Assistance','medium'),('Minimal Assistance','low')]}
}
SVG = {
 'bag': '<rect x="72" y="65" width="96" height="110" rx="18" fill="#7C3AED"/><path d="M96 65V48Q120 24 144 48V65" fill="none" stroke="#C4B5FD" stroke-width="9"/><rect x="87" y="102" width="66" height="52" rx="10" fill="#A78BFA"/>',
 'room': '<rect x="35" y="110" width="170" height="65" rx="12" fill="#2563EB"/><rect x="40" y="86" width="65" height="35" rx="9" fill="#93C5FD"/><rect x="110" y="86" width="90" height="35" rx="9" fill="#60A5FA"/>',
 'teeth': '<path d="M75 55 Q50 90 75 150 Q95 210 112 165 Q120 145 128 165 Q146 210 165 150 Q190 90 165 55 Q143 38 120 55 Q97 38 75 55Z" fill="#E2E8F0" stroke="#7C3AED" stroke-width="7"/>'
}

# Demo-only hash-linked local ledger. It is NOT a decentralized blockchain.
blockchain = []
received_ar_ids = set()
def calculate_hash(block):
    payload = json.dumps(block, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def add_block(task_id, level, source):
    record = {'index':len(blockchain)+1, 'task':task_id, 'status':'Completed', 'assistance_level':level,
              'source':source, 'timestamp':datetime.now(timezone.utc).isoformat(),
              'previous_hash':blockchain[-1]['hash'] if blockchain else '0'}
    record['hash'] = calculate_hash(record)
    blockchain.append(record)

def verify_blockchain():
    for i, block in enumerate(blockchain):
        fields = {k:v for k,v in block.items() if k != 'hash'}
        expected = blockchain[i-1]['hash'] if i else '0'
        if block.get('hash') != calculate_hash(fields) or block.get('previous_hash') != expected or block.get('index') != i+1:
            return False
    return True

css = '''
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&family=Inter:wght@400;500;700&display=swap');
body, .gradio-container {background:#0F172A!important;color:white!important;min-height:100vh;}
.gradio-container {max-width:850px!important;margin:auto!important;}
.gradio-container, .gradio-container * {font-family:'Tajawal','Inter',sans-serif!important;}
.gradio-container label,.gradio-container .prose,.gradio-container .prose h3 {color:white!important;}
.meawan-card {background:#1E293B;border:1px solid #334155;border-radius:18px;padding:24px;margin:14px 0;color:white;}
.meawan-card h1,.meawan-card h2,.meawan-card p {color:white!important;}
.meawan-card p {color:#CBD5E1!important;}
.meawan-header {text-align:center;}
.meawan-header h1 {font-size:42px;font-weight:800;margin:0;}
.step-card {text-align:center;}
.step-card h2 {font-size:28px;line-height:1.5;}
.progress-track {height:12px;border-radius:20px;overflow:hidden;background:#334155;margin-top:12px;}
.progress-fill {height:100%;background:#16A085;border-radius:20px;}
.primary-btn {background:#2563EB!important;color:white!important;border:0!important;border-radius:12px!important;min-height:56px!important;}
.next-btn,.verify-btn {background:#16A085!important;color:white!important;border:0!important;border-radius:12px!important;min-height:54px!important;}
.voice-btn {background:#7C3AED!important;color:white!important;border:0!important;border-radius:12px!important;min-height:54px!important;}
.ar-button {display:block;text-align:center;background:#7C3AED;color:white!important;padding:17px;border-radius:12px;text-decoration:none!important;font-weight:700;font-size:18px;}
.ar-button:focus {outline:3px solid #F59E0B;}
'''

def header(lang):
    t = TEXT[lang]; direction = 'rtl' if lang=='ar' else 'ltr'
    return f'<div class="meawan-card meawan-header" dir="{direction}"><h1>{t["title"]}</h1><p>{t["tagline"]}</p></div>'

def welcome(lang):
    t = TEXT[lang]; direction = 'rtl' if lang=='ar' else 'ltr'
    return f'<div class="meawan-card" dir="{direction}"><h2>{t["welcome"]}</h2><p>{t["intro"]}</p></div>'

def ar_link(lang):
    t=TEXT[lang]
    return f'<div class="meawan-card" dir="{"rtl" if lang=="ar" else "ltr"}"><h2>{t["ar_heading"]}</h2><a class="ar-button" href="{AR_URL}" target="_blank" rel="noopener noreferrer">{t["ar_open"]}</a></div>'

def display_step(lang, task_id, index, active, level, profile):
    t=TEXT[lang]; direction='rtl' if lang=='ar' else 'ltr'
    if not active:
        return f'<div class="meawan-card step-card" dir="{direction}"><p>{t["ready"]}</p></div>'
    steps=TASKS[task_id][lang]; total=len(steps); index=min(index,total)
    if index==total:
        return f'<div class="meawan-card step-card" dir="{direction}"><h2 style="color:#5EEAD4!important">{t["done"]}</h2><p>100%</p><div class="progress-track"><div class="progress-fill" style="width:100%"></div></div></div>'
    step_label=f'{t["step"]} {index+1} {t["of"]} {total}' if level!='low' else f'{index+1}/{total}'
    art=f'<svg viewBox="0 0 240 230" width="180" height="180" role="img" aria-label="{html.escape(steps[index])}">{SVG[task_id]}</svg>' if profile in ('simple','visual','motor') else ''
    pct=round(index*100/total)
    size='36px' if profile=='motor' else '28px'
    return f'<div class="meawan-card step-card" dir="{direction}"><p>{step_label}</p>{art}<h2 style="font-size:{size}">{html.escape(steps[index])}</h2><p>{t["progress"]}: {pct}%</p><div class="progress-track"><div class="progress-fill" style="width:{pct}%"></div></div></div>'

def start_task(lang, task_id, level, profile):
    state=(task_id,0,True,level)
    return state, display_step(lang,task_id,0,True,level,profile),gr.update(visible=True),gr.update(visible=True),gr.update(value=None,visible=False)

def next_step(lang, profile, state):
    task_id,index,active,level=state
    if not active:
        return state,display_step(lang,task_id,0,False,level,profile),gr.update(visible=False),gr.update(visible=False),gr.update(value=None,visible=False)
    total=len(TASKS[task_id][lang]); new_index=min(index+1,total)
    if index<total and new_index==total:
        add_block(task_id,level,'App')
    finished=new_index==total
    new_state=(task_id,new_index,True,level)
    return new_state,display_step(lang,task_id,new_index,True,level,profile),gr.update(visible=not finished),gr.update(visible=not finished),gr.update(value=None,visible=False)

def reset_task(lang,task_id,level,profile):
    return (task_id,0,False,level),display_step(lang,task_id,0,False,level,profile),gr.update(visible=False),gr.update(visible=False),gr.update(value=None,visible=False)

def refresh_step(lang,level,profile,state):
    task_id,index,active,saved_level=state
    return display_step(lang,task_id,index,active,saved_level if active else level,profile)

def change_language(lang,state,task_id,level,profile):
    t=TEXT[lang]; active_task,index,active,saved_level=state
    shown=active_task if active else task_id
    visible=active and index<len(TASKS[shown][lang])
    return (header(lang),welcome(lang),ar_link(lang),
            gr.update(choices=t['tasks'],label=t['task'],value=task_id),
            gr.update(choices=t['profiles'],label=t['profile'],value=profile),
            gr.update(choices=t['levels'],label=t['level'],value=level),
            gr.update(value=t['start']),gr.update(value=t['next'],visible=visible),
            gr.update(value=t['voice'],visible=visible),
            display_step(lang,shown,index,active,saved_level if active else level,profile),
            gr.update(value=None,visible=False,label=t['audio']),
            gr.update(value=f'### {t["history"]}'),gr.update(value=t['show']),gr.update(value=t['verify']),
            gr.update(label=t['records'],value=show_history(lang)),gr.update(label=t['result'],value=''))

def speak_instruction(lang,state):
    task_id,index,active,_=state
    if not active or index>=len(TASKS[task_id][lang]):
        return gr.update(value=None,visible=False)
    try:
        with tempfile.NamedTemporaryFile(suffix='.mp3',delete=False) as f:
            filename=f.name
        gTTS(text=TASKS[task_id][lang][index],lang=lang,slow=False).save(filename)
        return gr.update(value=filename,visible=True)
    except Exception:
        gr.Warning('تعذر إنشاء الصوت / Could not generate audio')
        return gr.update(value=None,visible=False)

def show_history(lang):
    t=TEXT[lang]
    if not blockchain: return t['empty']
    names=dict((value,label) for label,value in t['tasks'])
    levels=dict((value,label) for label,value in t['levels'])
    items=[]
    for b in reversed(blockchain):
        if lang=='ar':
            items.append(f'المهمة: {names.get(b["task"],b["task"])}\nالحالة: مكتملة\nالمصدر: {b["source"]}\nمستوى المساعدة: {levels.get(b["assistance_level"],b["assistance_level"])}\nالوقت: {b["timestamp"]}\nHash: {b["hash"][:24]}...')
        else:
            items.append(f'Task: {names.get(b["task"],b["task"])}\nStatus: Completed\nSource: {b["source"]}\nAssistance: {levels.get(b["assistance_level"],b["assistance_level"])}\nTime: {b["timestamp"]}\nHash: {b["hash"][:24]}...')
    return '\n\n--------------------\n\n'.join(items)

def check_records(lang):
    return TEXT[lang]['valid'] if verify_blockchain() else TEXT[lang]['invalid']

def receive_ar_completion(lang, request:gr.Request):
    params=dict(request.query_params) if request else {}
    task_id=params.get('ar_done','')
    completion_id=params.get('completion_id','')
    if task_id=='bag' and re.fullmatch(r'[0-9a-f]{32}',completion_id):
        if completion_id not in received_ar_ids:
            add_block('bag','high','AR')
            received_ar_ids.add(completion_id)
            message='تم تسجيل إنجاز AR في السجل التجريبي.' if lang=='ar' else 'AR completion was recorded in the demo ledger.'
        else:
            message='تم تسجيل هذا الإنجاز سابقًا.' if lang=='ar' else 'This completion was already recorded.'
    else:
        message='مرحبًا بك في مِعوان.' if lang=='ar' else 'Welcome to MEAWAN.'
    return f'<div class="meawan-card" dir="{"rtl" if lang=="ar" else "ltr"}"><p>{message}</p></div>',show_history(lang),check_records(lang)

with gr.Blocks(title='MEAWAN | مِعوان',css=css,theme=gr.themes.Base(primary_hue='blue',neutral_hue='slate')) as demo:
    state=gr.State(('bag',0,False,'high'))
    language=gr.Radio(choices=[('العربية','ar'),('English','en')],value='ar',label='Language / اللغة')
    title_html=gr.HTML(header('ar'))
    ar_status=gr.HTML('<div class="meawan-card">مرحبًا بك في مِعوان.</div>')
    welcome_html=gr.HTML(welcome('ar'))
    task=gr.Dropdown(choices=TEXT['ar']['tasks'],value='bag',label=TEXT['ar']['task'])
    profile=gr.Dropdown(choices=TEXT['ar']['profiles'],value='simple',label=TEXT['ar']['profile'])
    level=gr.Radio(choices=TEXT['ar']['levels'],value='high',label=TEXT['ar']['level'])
    start_btn=gr.Button(TEXT['ar']['start'],elem_classes='primary-btn')
    ar_html=gr.HTML(ar_link('ar'))
    step_display=gr.HTML(display_step('ar','bag',0,False,'high','simple'))
    with gr.Row():
        voice_btn=gr.Button(TEXT['ar']['voice'],elem_classes='voice-btn',visible=False)
        next_btn=gr.Button(TEXT['ar']['next'],elem_classes='next-btn',visible=False)
    audio_player=gr.Audio(label=TEXT['ar']['audio'],type='filepath',interactive=False,autoplay=False,visible=False)
    history_heading=gr.Markdown('### '+TEXT['ar']['history'])
    history_btn=gr.Button(TEXT['ar']['show'])
    history_output=gr.Textbox(label=TEXT['ar']['records'],lines=9,interactive=False)
    verify_btn=gr.Button(TEXT['ar']['verify'],elem_classes='verify-btn')
    verify_output=gr.Textbox(label=TEXT['ar']['result'],interactive=False)

    language.change(change_language,[language,state,task,level,profile],
        [title_html,welcome_html,ar_html,task,profile,level,start_btn,next_btn,voice_btn,step_display,audio_player,history_heading,history_btn,verify_btn,history_output,verify_output])
    start_btn.click(start_task,[language,task,level,profile],[state,step_display,next_btn,voice_btn,audio_player])
    next_btn.click(next_step,[language,profile,state],[state,step_display,next_btn,voice_btn,audio_player])
    voice_btn.click(speak_instruction,[language,state],audio_player)
    task.change(reset_task,[language,task,level,profile],[state,step_display,next_btn,voice_btn,audio_player])
    profile.change(refresh_step,[language,level,profile,state],step_display)
    level.change(refresh_step,[language,level,profile,state],step_display)
    history_btn.click(show_history,language,history_output)
    verify_btn.click(check_records,language,verify_output)
    demo.load(receive_ar_completion,inputs=[language],outputs=[ar_status,history_output,verify_output],queue=False)

demo.launch(share=True)

/tmp/ipykernel_6437/1387915275.py:189: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title='MEAWAN | مِعوان',css=css,theme=gr.themes.Base(primary_hue='blue',neutral_hue='slate')) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. [Errno 13] Permission denied: '.gradio'. This can happen when the current working directory is read-only.


In [3]:
import os

# Check AR target file
target_path = "/content/targets.mind"

if os.path.exists(target_path):
    print("AR target file is ready!")
else:
    print("AR target file not found!")

AR target file not found!


In [4]:
# MEAWAN - AR Web Page

import os
import shutil
from google.colab import files

project_dir = "/content/MEAWAN_AR"
os.makedirs(project_dir, exist_ok=True)

target_file = "/content/targets.mind"

if not os.path.exists(target_file):
    raise FileNotFoundError(
        "Please upload targets.mind to Colab first."
    )

shutil.copy(
    target_file,
    os.path.join(project_dir, "targets.mind")
)

ar_html = r"""
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport"
      content="width=device-width, initial-scale=1">

<title>MEAWAN AR</title>

<script src="https://aframe.io/releases/1.5.0/aframe.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/mind-ar@1.2.5/dist/mindar-image-aframe.prod.js"></script>

<style>
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;600;700;800&display=swap');

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    background: #0F172A;
    color: white;
    font-family: Tajawal, sans-serif;
}

#app {
    position: fixed;
    inset: 0;
    overflow: hidden;
}

a-scene {
    position: absolute !important;
    inset: 0;
}

.header {
    position: fixed;
    top: 12px;
    left: 12px;
    right: 12px;
    z-index: 10;
    background: rgba(15,23,42,.92);
    border: 1px solid #334155;
    border-radius: 16px;
    padding: 14px;
    text-align: center;
    pointer-events: none;
}

.header h1 {
    margin: 0;
    font-size: 25px;
}

.header p {
    margin: 4px 0;
    color: #CBD5E1;
    font-size: 13px;
}

.panel {
    position: fixed;
    bottom: 16px;
    left: 12px;
    right: 12px;
    z-index: 10;
    background: rgba(30,41,59,.96);
    border: 1px solid #334155;
    border-radius: 20px;
    padding: 20px;
    text-align: center;
}

#status {
    color: #C4B5FD;
    font-size: 14px;
}

#instruction {
    font-size: 25px;
    font-weight: 700;
    margin: 14px 0;
}

#progress {
    color: #CBD5E1;
}

button {
    width: 100%;
    min-height: 55px;
    border: none;
    border-radius: 12px;
    background: #2563EB;
    color: white;
    font-family: Tajawal, sans-serif;
    font-size: 19px;
    font-weight: 700;
    margin-top: 12px;
    cursor: pointer;
}

button:disabled {
    background: #475569;
}

.track {
    height: 9px;
    background: #334155;
    border-radius: 12px;
    overflow: hidden;
}

#bar {
    width: 0%;
    height: 100%;
    background: #16A085;
    transition: width .3s;
}
</style>
</head>

<body>
<div id="app">

    <a-scene
        mindar-image="imageTargetSrc: ./targets.mind;
                      autoStart: true;"
        color-space="sRGB"
        renderer="colorManagement: true"
        vr-mode-ui="enabled: false"
        device-orientation-permission-ui="enabled: false"
        embedded>

        <a-camera
            position="0 0 0"
            look-controls="enabled: false">
        </a-camera>

        <a-entity
            id="target"
            mindar-image-target="targetIndex: 0">

            <a-ring
                radius-inner="0.32"
                radius-outer="0.37"
                color="#16A085"
                position="0 0 0.03">
            </a-ring>

            <a-box
                position="0 0 0.15"
                width="0.25"
                height="0.25"
                depth="0.25"
                color="#7C3AED"
                animation="property: rotation;
                           to: 0 360 0;
                           loop: true;
                           dur: 3500;
                           easing: linear">
            </a-box>

        </a-entity>
    </a-scene>

    <div class="header">
        <h1>مِعوان | MEAWAN</h1>
        <p>مساعد الواقع المعزز</p>
    </div>

    <div class="panel">
        <div id="status">
            وجّه الكاميرا نحو صورة الحقيبة
        </div>

        <div id="instruction">
            افتح الحقيبة
        </div>

        <p id="progress">الخطوة 1 من 3</p>

        <div class="track">
            <div id="bar"></div>
        </div>

        <button id="next" onclick="nextStep()">
            أنجزت الخطوة
        </button>
    </div>

</div>

<script>
const steps = [
    "افتح الحقيبة",
    "ضع الكتاب داخل الحقيبة",
    "أغلق الحقيبة"
];

let currentStep = 0;

function nextStep() {
    currentStep++;

    const instruction =
        document.getElementById("instruction");

    const progress =
        document.getElementById("progress");

    const bar =
        document.getElementById("bar");

    const button =
        document.getElementById("next");

    if (currentStep >= steps.length) {
        instruction.textContent =
            "أحسنت! أكملت المهمة بنجاح.";

        progress.textContent = "100%";
        bar.style.width = "100%";
        button.textContent = "المهمة مكتملة";
        button.disabled = true;

        return;
    }

    instruction.textContent = steps[currentStep];

    progress.textContent =
        "الخطوة " + (currentStep + 1) +
        " من " + steps.length;

    bar.style.width =
        (currentStep / steps.length * 100) + "%";
}

document.addEventListener(
    "DOMContentLoaded",
    function() {
        const target =
            document.querySelector("#target");

        target.addEventListener(
            "targetFound",
            function() {
                document.getElementById(
                    "status"
                ).textContent =
                    "تم التعرف على صورة الحقيبة";
            }
        );

        target.addEventListener(
            "targetLost",
            function() {
                document.getElementById(
                    "status"
                ).textContent =
                    "وجّه الكاميرا نحو صورة الحقيبة";
            }
        );
    }
);
</script>
</body>
</html>
"""

html_path = os.path.join(
    project_dir, "index.html"
)

with open(
    html_path, "w", encoding="utf-8"
) as file:
    file.write(ar_html)

archive_path = shutil.make_archive(
    "/content/MEAWAN_AR",
    "zip",
    root_dir=project_dir
)

print("MEAWAN AR files are ready.")
print("HTML:", html_path)
print("Target:", project_dir + "/targets.mind")

files.download(archive_path)

ModuleNotFoundError: No module named 'google'

In [ ]:
# MEAWAN - Final AR to Blockchain Integration

import os
import shutil
from pathlib import Path
from google.colab import files

GRADIO_URL = "https://a3c35cb447255c8d0f.gradio.live"

project_dir = Path("/content/MEAWAN_AR")
html_path = project_dir / "index.html"
target_path = project_dir / "targets.mind"

# Check files
if not html_path.exists() or not target_path.exists():
    raise FileNotFoundError(
        "Run the previous AR creation cell first."
    )

page = html_path.read_text(encoding="utf-8")

# Insert achievement button
if 'id="saveAchievement"' not in page:

    old_button = """        </button>
    </div>"""

    new_button = """        </button>

        <button
            id="saveAchievement"
            onclick="saveAchievement()"
            style="
                display:none;
                background:#16A085;
                margin-top:12px;
            ">
            حفظ الإنجاز في مِعوان
        </button>
    </div>"""

    if old_button not in page:
        raise ValueError("AR button location not found")

    page = page.replace(
        old_button,
        new_button,
        1
    )

# Show save button after completing all steps
if 'document.getElementById("saveAchievement").style.display' not in page:

    old_completion = "button.disabled = true;"

    new_completion = """
        button.disabled = true;

        document.getElementById(
            "saveAchievement"
        ).style.display = "block";
    """

    if old_completion not in page:
        raise ValueError("Completion code not found")

    page = page.replace(
        old_completion,
        new_completion,
        1
    )

# Add connection to Gradio
if "function saveAchievement()" not in page:

    script = f"""
<script>
const MEAWAN_URL = "{GRADIO_URL}";

function saveAchievement() {{
    const completionId = crypto.randomUUID()
        .replace(/-/g, "");

    const url = new URL(MEAWAN_URL);

    url.searchParams.set("ar_done", "bag");
    url.searchParams.set(
        "completion_id",
        completionId
    );

    window.location.href = url.toString();
}}
</script>
"""

    page = page.replace(
        "</body>",
        script + "\n</body>",
        1
    )

# Save modified page
html_path.write_text(page, encoding="utf-8")

# Export updated AR website
zip_path = shutil.make_archive(
    "/content/MEAWAN_AR_Final",
    "zip",
    root_dir=str(project_dir)
)

print("AR integration file is ready.")
print("Gradio:", GRADIO_URL)
print("Blockchain return link configured.")

files.download(zip_path)

In [ ]:
# MEAWAN - AR Accessibility Improvements
# Visual arrows + Voice + Large buttons

from pathlib import Path
import shutil
from google.colab import files

GRADIO_URL = "https://a3c35cb447255c8d0f.gradio.live"

folder = Path("/content/MEAWAN_AR")
folder.mkdir(parents=True, exist_ok=True)

target = folder / "targets.mind"

if not target.exists():
    original = Path("/content/targets.mind")
    if original.exists():
        shutil.copy(original, target)
    else:
        raise FileNotFoundError("Upload targets.mind first")

page = r"""
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport"
 content="width=device-width, initial-scale=1">
<title>MEAWAN AR</title>

<script src="https://aframe.io/releases/1.5.0/aframe.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/mind-ar@1.2.5/dist/mindar-image-aframe.prod.js"></script>

<style>
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@400;500;700;800&display=swap');

* { box-sizing:border-box; }

body {
 margin:0;
 background:#0F172A;
 color:white;
 font-family:Tajawal,sans-serif;
}

a-scene {
 position:fixed!important;
 inset:0;
}

.header {
 position:fixed;
 top:12px;
 left:12px;
 right:12px;
 z-index:10;
 padding:15px;
 background:rgba(15,23,42,.93);
 border:1px solid #334155;
 border-radius:18px;
 text-align:center;
 pointer-events:none;
}

.header h1 {
 font-size:27px;
 margin:0;
}

.header p {
 color:#CBD5E1;
 margin:5px 0 0;
}

.panel {
 position:fixed;
 bottom:12px;
 left:12px;
 right:12px;
 z-index:10;
 padding:18px;
 background:rgba(30,41,59,.96);
 border:1px solid #475569;
 border-radius:20px;
 text-align:center;
 max-height:55vh;
 overflow:auto;
}

#status {
 color:#5EEAD4;
 font-size:16px;
}

#instruction {
 font-size:28px;
 line-height:1.5;
 margin:12px 0;
}

#hint {
 color:#CBD5E1;
 font-size:18px;
 line-height:1.6;
}

#progress {
 color:#C4B5FD;
 font-size:17px;
}

.track {
 background:#334155;
 height:10px;
 border-radius:12px;
 overflow:hidden;
 margin:12px 0;
}

#bar {
 background:#16A085;
 height:100%;
 width:0%;
 transition:width .3s;
}

button {
 width:100%;
 min-height:62px;
 border:0;
 border-radius:12px;
 color:white;
 font-size:20px;
 font-weight:700;
 font-family:Tajawal,sans-serif;
 margin-top:10px;
 cursor:pointer;
}

.tools {
 display:grid;
 grid-template-columns:1fr 1fr;
 gap:10px;
}

#listen { background:#7C3AED; }
#resize { background:#475569; }
#next { background:#16A085; }
#save {
 background:#2563EB;
 display:none;
}

button:disabled {
 opacity:.65;
}

body.large #instruction { font-size:38px; }
body.large #hint { font-size:25px; }
body.large button {
 font-size:24px;
 min-height:78px;
}

@media(max-width:400px) {
 .panel { padding:12px; }
 .header h1 { font-size:23px; }
}
</style>
</head>

<body>

<a-scene
 mindar-image="imageTargetSrc: ./targets.mind; autoStart: true;"
 color-space="sRGB"
 renderer="colorManagement: true"
 vr-mode-ui="enabled: false"
 device-orientation-permission-ui="enabled: false"
 embedded>

 <a-camera position="0 0 0"
   look-controls="enabled:false">
 </a-camera>

 <a-entity id="target"
   mindar-image-target="targetIndex:0">

   <a-entity id="arrow"
     position="0 0 0.12"
     rotation="0 0 180"
     animation="property: position;
                to: 0 -0.08 0.12;
                dur: 900;
                dir: alternate;
                loop: true;
                easing: easeInOutSine">

     <a-box
       position="0 -0.05 0"
       width="0.10"
       height="0.30"
       depth="0.04"
       color="#7C3AED">
     </a-box>

     <a-triangle
       vertex-a="0 0.20 0"
       vertex-b="-0.21 -0.08 0"
       vertex-c="0.21 -0.08 0"
       position="0 0.14 0"
       material="color:#16A085; side:double">
     </a-triangle>

   </a-entity>

 </a-entity>
</a-scene>

<div class="header">
 <h1>مِعوان | MEAWAN</h1>
 <p>مساعد الواقع المعزز</p>
</div>

<div class="panel">
 <div id="status" role="status">
  وجّه الكاميرا نحو صورة الحقيبة
 </div>

 <h2 id="instruction">افتح الحقيبة</h2>

 <p id="hint">
  ابدأ بفتح السحاب برفق.
 </p>

 <p id="progress">الخطوة 1 من 3</p>

 <div class="track">
  <div id="bar"></div>
 </div>

 <div class="tools">
  <button id="listen" onclick="speak()">
   استمع للتعليمات
  </button>

  <button id="resize" onclick="resizeText()">
   تكبير الخط
  </button>
 </div>

 <button id="next" onclick="nextStep()">
  أنجزت الخطوة
 </button>

 <button id="save" onclick="saveAchievement()">
  حفظ الإنجاز في مِعوان
 </button>
</div>

<script>
const GRADIO_URL = "__GRADIO_URL__";

const steps = [
 {
  text:"افتح الحقيبة",
  hint:"افتح سحاب الحقيبة برفق.",
  x:0.0,
  y:0.34,
  rotation:180
 },
 {
  text:"ضع الكتاب داخل الحقيبة",
  hint:"خذ الكتاب وضعه داخل الحقيبة.",
  x:-0.22,
  y:0.05,
  rotation:90
 },
 {
  text:"أغلق الحقيبة",
  hint:"أغلق سحاب الحقيبة بعد وضع الكتاب.",
  x:0.10,
  y:0.32,
  rotation:180
 }
];

let currentStep = 0;

function updateStep() {
 const step = steps[currentStep];

 document.getElementById("instruction")
  .textContent = step.text;

 document.getElementById("hint")
  .textContent = step.hint;

 document.getElementById("progress")
  .textContent =
   "الخطوة " + (currentStep+1) + " من 3";

 document.getElementById("bar")
  .style.width =
   (currentStep/steps.length*100)+"%";

 const arrow = document.getElementById("arrow");

 arrow.setAttribute(
  "position",
  `${step.x} ${step.y} 0.12`
 );

 arrow.setAttribute(
  "rotation",
  `0 0 ${step.rotation}`
 );

 // Remove any old position animation
 arrow.removeAttribute("animation");

 arrow.setAttribute(
  "animation",
  `property: position;
   from: ${step.x} ${step.y} 0.12;
   to: ${step.x} ${step.y-0.08} 0.12;
   dur: 900;
   dir: alternate;
   loop: true`
 );
}

function nextStep() {
 if(currentStep >= steps.length) return;

 window.speechSynthesis?.cancel();

 currentStep++;

 if(currentStep >= steps.length) {
  document.getElementById("instruction")
   .textContent="أحسنت! أكملت المهمة بنجاح.";

  document.getElementById("hint")
   .textContent="يمكنك الآن حفظ إنجازك.";

  document.getElementById("progress")
   .textContent="100%";

  document.getElementById("bar")
   .style.width="100%";

  document.getElementById("next")
   .style.display="none";

  document.getElementById("save")
   .style.display="block";

  document.getElementById("arrow")
   .setAttribute("visible",false);

  return;
 }

 updateStep();
}

function speak() {
 if(!("speechSynthesis" in window)) {
  document.getElementById("status")
   .textContent="الصوت غير متاح في هذا المتصفح";
  return;
 }

 window.speechSynthesis.cancel();

 const step=steps[currentStep];
 if(!step) return;

 const voice = new SpeechSynthesisUtterance(
  step.text + ". " + step.hint
 );

 voice.lang="ar-SA";
 voice.rate=0.85;
 voice.pitch=1;

 window.speechSynthesis.speak(voice);
}

function resizeText() {
 document.body.classList.toggle("large");

 document.getElementById("resize")
  .textContent=document.body.classList.contains("large")
   ? "الحجم العادي" : "تكبير الخط";
}

function saveAchievement() {
 if(currentStep < steps.length) return;

 const completionId = crypto.randomUUID()
  .replace(/-/g,"");

 const url = new URL(GRADIO_URL);

 url.searchParams.set("ar_done","bag");
 url.searchParams.set("completion_id",completionId);

 window.location.href=url.toString();
}

document.addEventListener("DOMContentLoaded",()=>{
 const target=document.getElementById("target");

 target.addEventListener("targetFound",()=>{
  document.getElementById("status")
   .textContent="تم التعرف على صورة الحقيبة";
 });

 target.addEventListener("targetLost",()=>{
  document.getElementById("status")
   .textContent="وجّه الكاميرا نحو صورة الحقيبة";
 });

 updateStep();
});
</script>
</body>
</html>
"""

page = page.replace("__GRADIO_URL__", GRADIO_URL)

(folder / "index.html").write_text(
    page, encoding="utf-8"
)

zip_path = shutil.make_archive(
    "/content/MEAWAN_AR_Improved",
    "zip",
    root_dir=str(folder)
)

print("Improved MEAWAN AR is ready.")
files.download(zip_path)